Notes: NWM data for Hawaii was ingested using two different variables names (`streamflow_15min_inst`, `streamflow_hourly_inst`)

This just drops duplicate rows across both variable names for the `nwm30_analysis_assim_hawaii_no_da` configuration. There are none for HI short range.

Any mis-named variables for these configurations will then be updated to `streamflow_15min_inst` using a migration

In [ ]:
import os
from pathlib import Path
import tempfile
import shutil

import teehr
from teehr.models.filters import TableFilter
from teehr.evaluation.spark_session_utils import create_spark_session

from pyspark.sql import functions as F
from pyspark.sql.window import Window

teehr.__version__

In [ ]:
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=8,
    executor_memory="32g",
    executor_cores=4,
    aws_profile="admin-user"
)

In [ ]:
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

In [ ]:
config_name = "nwm30_analysis_assim_hawaii_no_da"

filters = [
    TableFilter(
        column='configuration_name',
        operator='=',
        value=config_name
    )
]

source = ev.secondary_timeseries.filter(filters).to_sdf()    

In [ ]:
total_rows = source.count()
print(f"Total num rows before: {total_rows}")

In [ ]:
group_by_cols = ["value_time", "reference_time", "unit_name", "location_id", "value"]

window_spec = Window.partitionBy(group_by_cols).orderBy("variable_name")
source_with_rank = source.withColumn("row_num", F.row_number().over(window_spec))

In [ ]:
# Get the number of duplicates
num_dups = source_with_rank.filter("row_num == 2").count() 
print(f"Number of dups across variable_name: {num_dups}")

In [ ]:
# Filter out duplicates
nonoverlapping_sdf = source_with_rank.filter("row_num == 1").drop("row_num")

num_rows_after = nonoverlapping_sdf.count()
nonoverlapping_sdf.show()

In [ ]:
print(f"Total num rows before: {total_rows}")
print(f"Total num rows after: {num_rows_after}")
print(f"Rows removed: {total_rows - num_rows_after}")

In [ ]:
# Optional preview of rows that will be deleted
rows_to_delete = ev.secondary_timeseries.delete(
    filters=filters,
    dry_run=True,
)
print(f"rows to delete: {rows_to_delete.count()}")

In [ ]:
# Delete the bad slice
deleted_count = ev.secondary_timeseries.delete(filters=filters)
print(f"deleted rows: {deleted_count}")

In [ ]:
# Insert the cleaned slice back
ev.secondary_timeseries.load_dataframe(
    nonoverlapping_sdf,
    write_mode="append",
    drop_duplicates=True,
)

In [ ]:
ev.secondary_timeseries.filter(filters).to_sdf().count()   